# 3 · Probabilités et mesure de la qualité — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch03_probabilites/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 3.1, 3.2, 3.3, 3.5, 3.6, 3.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 3.12 à 3.14 | probabilités par les aires et les tables : fléchettes simulées, deux disques, pd.crosstab (*prochaine session de génération*) | | | |
| B | 3.15 à 3.19 | la matrice de confusion et les mesures binaires dans mylearn.metrics (*prochaine session de génération*) | | | |
| C | 3.20 à 3.23 | seuil de décision, prévalence, outils et documentation de scikit-learn (*prochaine session de génération*) | | | |
| D | 3.24 à 3.29 | au-delà du livre : courbes ROC et precision-recall, moyennes sur plusieurs classes, calibration, défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer des probabilités simples, conditionnelles, jointes et marginales, et ne pas confondre $P(A \mid B)$ avec $P(B \mid A)$.
- Construire une matrice de confusion et en tirer accuracy, precision, recall, F1 et les autres mesures, avec les conventions de scikit-learn.
- Choisir une mesure et un seuil selon le coût des erreurs et la prévalence.
- Évaluer des scores sur tous les seuils (courbes ROC et precision-recall) et vérifier des probabilités (calibration).

**Rappel express.** $P(A \mid B) = \frac{P(A, B)}{P(B)}$ ; $P(A, B) = P(A \mid B)\,P(B)$ ; matrice de confusion de scikit-learn pour des étiquettes 0/1 : `[[TN, FP], [FN, TP]]` ; accuracy $= \frac{TP + TN}{n}$ ; precision $= \frac{TP}{TP + FP}$ ; recall $= \frac{TP}{TP + FN}$ ; spécificité $= \frac{TN}{TN + FP}$ ; $F_1 = \frac{2\,TP}{2\,TP + FP + FN}$. En Python : `pd.crosstab`, `sklearn.metrics.confusion_matrix`, `classification_report`, `roc_curve`, `precision_recall_curve`.

## Partie 0 · Vérifier tes exercices papier (✏️ 3.1, 3.2, 3.3, 3.5, 3.6 et 3.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 3.4 et 3.8 se corrigent avec `05_solutions.md`.

In [2]:
# The paper exercises only use the numbers of their statements (02_exercices.md)
import numpy as np

WALL = 4 * 2.5                                     # 3.1: area of the wall (m²)
AREA_A, AREA_B, AREA_AB = 2 * 1.5, 2 * 1, 1 * 1    # 3.1: rectangles A, B and their overlap
TRUTH_32 = np.array([0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0])   # 3.2
PRED_32 = np.array([0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0])
TP_32 = int(np.sum((TRUTH_32 == 1) & (PRED_32 == 1)))
FP_32 = int(np.sum((TRUTH_32 == 0) & (PRED_32 == 1)))
FN_32 = int(np.sum((TRUTH_32 == 1) & (PRED_32 == 0)))
TN_32 = int(np.sum((TRUTH_32 == 0) & (PRED_32 == 0)))
ICE = np.array([[42, 18], [38, 52]])              # 3.3: rows vanilla, chocolate; columns cone, cup
TP_35, FN_35, FP_35, TN_35 = 40, 10, 60, 890      # 3.5: fraud detector, 1 000 transactions
SPECIES_36 = np.array([[41, 3, 1], [7, 13, 0], [0, 2, 33]])   # 3.6: rows truth, columns prediction


def rates(tp, fn, fp, tn):
    """Every measure of a binary confusion matrix (3.5)."""
    n = tp + fn + fp + tn
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    specificity, npv = tn / (tn + fp), tn / (tn + fn)
    return {"prevalence": (tp + fn) / n, "accuracy": (tp + tn) / n, "precision": precision, "recall": recall,
            "specificity": specificity, "npv": npv, "fpr": fp / (fp + tn), "fnr": fn / (fn + tp),
            "fdr": fp / (fp + tp), "for": fn / (fn + tn), "f1": 2 * tp / (2 * tp + fp + fn),
            "balanced": (recall + specificity) / 2,
            "mcc": (tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))}


R35 = rates(TP_35, FN_35, FP_35, TN_35)
PRECISION_36 = np.diag(SPECIES_36) / SPECIES_36.sum(axis=0)
RECALL_36 = np.diag(SPECIES_36) / SPECIES_36.sum(axis=1)
F1_36 = 2 * PRECISION_36 * RECALL_36 / (PRECISION_36 + RECALL_36)
SUPPORT_36 = SPECIES_36.sum(axis=1)


def screening(n, prevalence, sensitivity, specificity):
    """Expected counts of a screening test (natural frequencies): TP, FN, FP, TN (3.7)."""
    sick = n * prevalence
    healthy = n - sick
    return sensitivity * sick, (1 - sensitivity) * sick, (1 - specificity) * healthy, specificity * healthy


TP_37, FN_37, FP_37, TN_37 = screening(50_000, 0.01, 0.99, 0.99)
BOOK = screening(10_000, 0.01, 0.99, 0.98)        # 3.7 part 2: the book's test (§3.8)

**Ex 3.1 — Fléchettes et aires : probabilités simples et conditionnelles**

In [3]:
wb.record("3.1a", AREA_A / WALL, decimals=2, mistakes={"c'est l'aire de A : divise-la par l'aire du mur": AREA_A})
wb.record("3.1b", AREA_B / WALL, decimals=2, mistakes={"c'est l'aire de B : divise-la par l'aire du mur": AREA_B})
wb.record("3.1c", AREA_AB / WALL, decimals=2, mistakes={"c'est P(A) × P(B) : ce produit ne vaut P(A, B) que si A et B sont indépendants ; mesure plutôt l'aire de la partie commune": (AREA_A / WALL) * (AREA_B / WALL), "c'est l'aire de la partie commune : divise-la par l'aire du mur": AREA_AB, "vérifie les bornes de la partie commune : pour chaque coordonnée, garde le plus grand des deux minimums et le plus petit des deux maximums": 1 * 1.5 / WALL})
wb.record("3.1d", AREA_AB / AREA_B, decimals=2, mistakes={"c'est P(B | A) : ici, on sait que la fléchette est dans B, divise par l'aire de B": AREA_AB / AREA_A, "c'est P(A, B) : sachant B, on divise par l'aire de B, pas par celle du mur": AREA_AB / WALL})
wb.record("3.1e", AREA_AB / AREA_A, decimals=3, mistakes={"c'est P(A | B) : ici, on sait que la fléchette est dans A, divise par l'aire de A": AREA_AB / AREA_B, "c'est P(A, B) : sachant A, on divise par l'aire de A, pas par celle du mur": AREA_AB / WALL})
wb.record("3.1f", 1 - (AREA_A + AREA_B - AREA_AB) / WALL, decimals=2, mistakes={"la partie commune est comptée deux fois dans P(A) + P(B) : P(A ou B) = P(A) + P(B) − P(A, B)": 1 - (AREA_A + AREA_B) / WALL, "c'est P(A ou B) : on demande l'événement contraire, ni A ni B": (AREA_A + AREA_B - AREA_AB) / WALL, "(1 − P(A)) × (1 − P(B)) n'est juste que si A et B sont indépendants, ce que rien ne garantit ici (tu le vérifieras en j) : passe par P(A ou B)": (1 - AREA_A / WALL) * (1 - AREA_B / WALL)})
wb.record("3.1g", round(400 * AREA_B / WALL), mistakes={"c'est le nombre attendu dans A et B à la fois": round(400 * AREA_AB / WALL), "c'est le nombre attendu dans A": round(400 * AREA_A / WALL)})
wb.record("3.1h", round(400 * AREA_AB / WALL), mistakes={"c'est le nombre attendu dans B : on demande A et B à la fois": round(400 * AREA_B / WALL)})
wb.record("3.1i", 41 / 76, decimals=3, mistakes={"sachant B : divise par le nombre de fléchettes dans B, pas par toutes": 41 / 400, "divise les fléchettes dans A et B par celles dans B, pas l'inverse": 76 / 41})
wb.record("3.1j", bool(np.isclose(AREA_AB / WALL, (AREA_A / WALL) * (AREA_B / WALL))), mistakes={"compare P(A, B) au produit P(A) × P(B)": True})

WB_ANSWER {"decimals": 2, "hash": "adee4b45bf32a79824d87508647ff796f1cd5c87b1b7e65863f7dcfe8d1202e6", "hash_coarse": "a2162fa7d213ddd7f0b2bf4d3fa9e192d6b29183236b3bca2f0ddd1b0e076e4a", "id": "3.1a", "kind": "float", "magnitude": -1, "mistakes": {"e7903c20aa76e96ea32532d84e4d62c3366648d0cc1d0bf75264596fd3ed112f": "c'est l'aire de A : divise-la par l'aire du mur"}, "sign": 1}
📝 Ex 3.1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "796049172a19f6375a44c96deb6acc722a106c1f94d8ca1f2f565c47eb4e456b", "hash_coarse": "899c0a59a2a06ecef02be72b9a22e221789801b67b664475750ac89b460f2988", "id": "3.1b", "kind": "float", "magnitude": -1, "mistakes": {"046221996f883b39e2eddd4e0f69a383dc20cf2c21a68d738a6bb1d0ad845d80": "c'est l'aire de B : divise-la par l'aire du mur"}, "sign": 1}
📝 Ex 3.1b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "9678a63db93db5e80edfe0ef04ec56d1167b339d3133d363f00f078cb7aa1038", "hash_coarse": "91c4e8ac33f7722e

**Ex 3.2 — Les 20 points : matrice de confusion et quatre mesures**

In [4]:
wb.record("3.2a", TP_32, mistakes={"ça, c'est le nombre de faux positifs : un vrai positif est positif ET prédit positif": FP_32, "c'est le nombre total de positifs : ne compte que ceux qui sont aussi prédits positifs": 10, "ça, c'est le nombre de vrais négatifs : un vrai positif est positif ET prédit positif": TN_32})
wb.record("3.2b", FP_32, mistakes={"ça, c'est le nombre de faux négatifs : un faux positif est un NÉGATIF prédit positif": FN_32})
wb.record("3.2c", FN_32, mistakes={"ça, c'est le nombre de faux positifs : un faux négatif est un POSITIF prédit négatif": FP_32})
wb.record("3.2d", TN_32, mistakes={"c'est le nombre total de négatifs : ne compte que ceux qui sont aussi prédits négatifs": 10, "ça, c'est le nombre de vrais positifs : un vrai négatif est négatif ET prédit négatif": TP_32})
wb.record("3.2e", [[TN_32, FP_32], [FN_32, TP_32]], mistakes={"c'est la disposition du livre (TP en haut à gauche) ; scikit-learn trie les étiquettes, 0 puis 1, donc la ligne de la vérité 0 vient d'abord": [[TP_32, FN_32], [FP_32, TN_32]], "vérité en LIGNES, prédiction en COLONNES : ta matrice est transposée": [[TN_32, FN_32], [FP_32, TP_32]]})
wb.record("3.2f", (TP_32 + TN_32) / 20, decimals=2, mistakes={"l'accuracy compte les DEUX sortes de bonnes réponses, TP et TN": TP_32 / 20, "c'est le recall : l'accuracy compte les bonnes réponses, TP + TN, parmi les 20 points": TP_32 / (TP_32 + FN_32), "c'est la precision : l'accuracy compte les bonnes réponses, TP + TN, parmi les 20 points": TP_32 / (TP_32 + FP_32)})
wb.record("3.2g", TP_32 / (TP_32 + FP_32), decimals=3, mistakes={"c'est le recall : la precision divise par le nombre de prédictions positives, TP + FP": TP_32 / (TP_32 + FN_32)})
wb.record("3.2h", TP_32 / (TP_32 + FN_32), decimals=2, mistakes={"c'est la precision : le recall divise par le nombre de positifs réels, TP + FN": TP_32 / (TP_32 + FP_32), "c'est l'accuracy : le recall ne regarde que les positifs réels, TP + FN": (TP_32 + TN_32) / 20})
wb.record("3.2i", 2 * TP_32 / (2 * TP_32 + FP_32 + FN_32), decimals=3, mistakes={"c'est la moyenne ordinaire de la precision et du recall : le F1 est leur moyenne HARMONIQUE": (TP_32 / (TP_32 + FP_32) + TP_32 / (TP_32 + FN_32)) / 2, "deux causes possibles : tu as calculé avec la precision arrondie à 3 décimales, ou tu as tronqué au lieu d'arrondir au plus proche ; garde la fraction exacte jusqu'au bout (ou la forme 2 TP / (2 TP + FP + FN)), puis arrondis au plus proche": 0.666})

WB_ANSWER {"hash": "cd099ca46bbf1ec63eb2d7a535c8b237c62537a88a042f2235ff6d777db48a11", "id": "3.2a", "kind": "int", "magnitude": 0, "mistakes": {"8b083a362cc7162cd36a09bcdf4ce97cc93054277010b9f2688ea80cb91c466e": "ça, c'est le nombre de faux positifs : un vrai positif est positif ET prédit positif", "9c0fbc2400b50e3098841e8ec0b121ae531845c4f6e44641171370b3e5626d72": "c'est le nombre total de positifs : ne compte que ceux qui sont aussi prédits positifs", "c6e0a5708efb0ee04de60a6b75ceaf9f134f4f9f475f6f179c1934f2933c0b26": "ça, c'est le nombre de vrais négatifs : un vrai positif est positif ET prédit positif"}, "sign": 1}
📝 Ex 3.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "af62043631e0148bdc88d6da2f5e8cc3d8e0e338cc7156ed93ba98e7e3b80241", "id": "3.2b", "kind": "int", "magnitude": 0, "mistakes": {"4f94137f77cbb27c4720366e0fb452c36fb779f513fe1c43c44e7941cc0112d6": "ça, c'est le nombre de faux négatifs : un faux positif est un NÉGATIF prédit positif"}, "sign": 1}
📝 Ex 3.2b : réponse 

**Ex 3.3 — Le glacier : jointes, marginales et conditionnelles**

In [5]:
wb.record("3.3a", ICE[0].sum() / ICE.sum(), decimals=2, mistakes={"c'est un effectif : divise par le nombre de clients": int(ICE[0].sum())})
wb.record("3.3b", ICE[:, 0].sum() / ICE.sum(), decimals=3, mistakes={"c'est la probabilité d'un pot : C est l'événement « cornet »": ICE[:, 1].sum() / ICE.sum()})
wb.record("3.3c", ICE[0, 0] / ICE.sum(), decimals=2, mistakes={"c'est P(V | C) : la probabilité jointe divise par le nombre total de clients": ICE[0, 0] / ICE[:, 0].sum(), "c'est P(C | V) : la probabilité jointe divise par le nombre total de clients": ICE[0, 0] / ICE[0].sum(), "c'est P(V) × P(C) : ce produit ne vaut P(V, C) que si V et C sont indépendants ; lis plutôt la case de la table": (ICE[0].sum() / ICE.sum()) * (ICE[:, 0].sum() / ICE.sum())})
wb.record("3.3d", ICE[0, 0] / ICE[:, 0].sum(), decimals=3, mistakes={"c'est P(C | V) : sachant C (un cornet), divise par le nombre de cornets": ICE[0, 0] / ICE[0].sum(), "c'est P(V, C) : sachant C, divise par le nombre de cornets, pas de clients": ICE[0, 0] / ICE.sum()})
wb.record("3.3e", ICE[0, 0] / ICE[0].sum(), decimals=2, mistakes={"c'est P(V | C) : sachant V (vanille), divise par le nombre de clients vanille": ICE[0, 0] / ICE[:, 0].sum(), "c'est P(V, C) : sachant V, divise par le nombre de clients vanille": ICE[0, 0] / ICE.sum()})
wb.record("3.3f", ICE[1, 1] / ICE[:, 1].sum(), decimals=3, mistakes={"c'est P(pot | chocolat) : sachant « pot », divise par le nombre de pots": ICE[1, 1] / ICE[1].sum(), "c'est la probabilité jointe : sachant « pot », divise par le nombre de pots": ICE[1, 1] / ICE.sum()})
wb.record("3.3g", bool(np.isclose(ICE[0, 0] / ICE.sum(), (ICE[0].sum() / ICE.sum()) * (ICE[:, 0].sum() / ICE.sum()))), mistakes={"compare P(V, C) au produit P(V) × P(C)": True})
wb.record("3.3h", round(300 * ICE[0, 0] / ICE.sum()), mistakes={"c'est le nombre d'hier, pour 150 clients : demain, ils seront 300": int(ICE[0, 0])})

WB_ANSWER {"decimals": 2, "hash": "367e2c491c5a1c543dd801cdb1b8747af099c3e37517306472d94365809100b8", "hash_coarse": "e7d9fd84dde728efbbc68d1781824f19d4443882c7c0f09ebadc484c51efa53d", "id": "3.3a", "kind": "float", "magnitude": -1, "mistakes": {"cc9bd2866713a932cb937fe631eb400b0ed95ae855923a8cb0b83d8ee1dc58f6": "c'est un effectif : divise par le nombre de clients"}, "sign": 1}
📝 Ex 3.3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "228b5dac8b4547819d4e56c1b9e9b69a1bfbb63b534da2ca7a5d3fbf2ccbaab2", "hash_coarse": "2773d902f96ea74b7dd8dd110e9269a2e054acc64d4842cec8667c70d64b0634", "id": "3.3b", "kind": "float", "magnitude": -1, "mistakes": {"52ba041dfd5596fa9ff6b527cd0a5a7d17a43fed34e260bcda872fd394b0b16d": "c'est la probabilité d'un pot : C est l'événement « cornet »"}, "sign": 1}
📝 Ex 3.3b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "0b199dfe47066f392dda35d34c2a9f4c01784dcfddef076636b4c61f6ab5ee40", "hash_coarse": 

**Ex 3.5 — Toutes les mesures du tableau récapitulatif**

In [6]:
wb.record("3.5a", R35["prevalence"], decimals=3, mistakes={"c'est la part de prédictions positives : la prévalence compte les positifs réels (la vérité), TP + FN": (TP_35 + FP_35) / 1000})
wb.record("3.5b", R35["accuracy"], decimals=3, mistakes={"l'accuracy compte les DEUX sortes de bonnes réponses, TP et TN": TP_35 / 1000})
wb.record("3.5c", R35["precision"], decimals=3, mistakes={"c'est le recall : la precision divise par les prédictions positives, TP + FP": R35["recall"]})
wb.record("3.5d", R35["recall"], decimals=3, mistakes={"c'est la precision : le recall divise par les positifs réels, TP + FN": R35["precision"]})
wb.record("3.5e", R35["specificity"], decimals=3, mistakes={"c'est la NPV : la spécificité divise par le nombre de négatifs RÉELS, TN + FP": R35["npv"], "c'est le FPR, son complément à 1": R35["fpr"]})
wb.record("3.5f", R35["npv"], decimals=3, mistakes={"c'est la spécificité : la NPV divise par le nombre de prédictions négatives, TN + FN": R35["specificity"], "c'est le FOR, son complément à 1": R35["for"]})
wb.record("3.5g", R35["fpr"], decimals=3, mistakes={"c'est le FDR : le FPR divise par le nombre de négatifs réels, FP + TN": R35["fdr"], "c'est la spécificité, son complément à 1": R35["specificity"], "tu as divisé par les 1 000 transactions : le FPR divise par les négatifs réels, FP + TN": FP_35 / 1000})
wb.record("3.5h", R35["fnr"], decimals=3, mistakes={"c'est le FOR : le FNR divise par le nombre de positifs réels, FN + TP": R35["for"], "c'est le recall, son complément à 1": R35["recall"]})
wb.record("3.5i", R35["fdr"], decimals=3, mistakes={"c'est le FPR : le FDR divise par le nombre de prédictions positives, FP + TP": R35["fpr"], "c'est la precision, son complément à 1": R35["precision"]})
wb.record("3.5j", R35["for"], decimals=3, mistakes={"c'est le FNR : le FOR divise par le nombre de prédictions négatives, FN + TN": R35["fnr"], "c'est la NPV, son complément à 1": R35["npv"], "tu as divisé par les 1 000 transactions : le FOR divise par les prédictions négatives, FN + TN": FN_35 / 1000})
wb.record("3.5k", R35["f1"], decimals=3, mistakes={"c'est la moyenne ordinaire de la precision et du recall : le F1 est leur moyenne HARMONIQUE": (R35["precision"] + R35["recall"]) / 2})
wb.record("3.5l", R35["balanced"], decimals=3, mistakes={"c'est l'accuracy : la balanced accuracy est la moyenne du recall et de la spécificité": R35["accuracy"], "tu as fait la moyenne avec une spécificité déjà arrondie : garde la fraction exacte jusqu'au bout, puis arrondis": 0.869})
wb.record("3.5m", R35["mcc"], decimals=3, mistakes={"le dénominateur est la RACINE CARRÉE du produit des quatre sommes": (TP_35 * TN_35 - FP_35 * FN_35) / ((TP_35 + FP_35) * (TP_35 + FN_35) * (TN_35 + FP_35) * (TN_35 + FN_35))})

WB_ANSWER {"decimals": 3, "hash": "7d20e31ab551298d68b98513e37e79243d53260c946da7c24a29c5bc4940944d", "hash_coarse": "b312899b99ac63b886a6519f98d117a998db6dc15c605fe7ebadaf26dd17aec8", "id": "3.5a", "kind": "float", "magnitude": -2, "mistakes": {"eaa3d47c0504b18aafe8c0573288b040c9f400fa216dc6aab106263da1ea6dbb": "c'est la part de prédictions positives : la prévalence compte les positifs réels (la vérité), TP + FN"}, "sign": 1}
📝 Ex 3.5a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "fa8e5784ac97ddd5f265053b49a859298394c67e87ca6bc239e9817bd3a51a6c", "hash_coarse": "0f0b925ff46ab5a84725c1170bbb217114806f9c3b97cb2222ce1bd512c0ad33", "id": "3.5b", "kind": "float", "magnitude": -1, "mistakes": {"624df273df64bdbe7597d00350850ba6012e84ddf1699662513b229126d9703e": "l'accuracy compte les DEUX sortes de bonnes réponses, TP et TN"}, "sign": 1}
📝 Ex 3.5b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "df97f1c13ef8782c4e7ca6ece97168

**Ex 3.6 — Trois espèces : moyennes macro, micro et pondérée**

In [7]:
wb.record("3.6a", SUPPORT_36.tolist(), mistakes={"ce sont les nombres de PRÉDICTIONS de chaque espèce (totaux des colonnes) : le support compte les vrais manchots de chaque espèce (totaux des lignes)": SPECIES_36.sum(axis=0).tolist()})
wb.record("3.6b", PRECISION_36.tolist(), decimals=3, mistakes={"ce sont les recalls : la precision d'une espèce divise par le total de sa COLONNE (les prédictions)": RECALL_36.tolist()})
wb.record("3.6c", RECALL_36.tolist(), decimals=3, mistakes={"ce sont les precisions : le recall d'une espèce divise par le total de sa LIGNE (les vrais manchots)": PRECISION_36.tolist()})
wb.record("3.6d", F1_36.tolist(), decimals=3, mistakes={"ce sont des moyennes ordinaires : le F1 est la moyenne HARMONIQUE de la precision et du recall": ((PRECISION_36 + RECALL_36) / 2).tolist()})
wb.record("3.6e", PRECISION_36.mean(), decimals=3, mistakes={"c'est la precision pondérée : la moyenne macro est la moyenne SIMPLE des trois espèces": np.average(PRECISION_36, weights=SUPPORT_36), "c'est la precision micro, qui vaut l'accuracy": np.trace(SPECIES_36) / SPECIES_36.sum()})
wb.record("3.6f", F1_36.mean(), decimals=3, mistakes={"c'est le F1 pondéré : la moyenne macro est la moyenne SIMPLE des trois espèces": np.average(F1_36, weights=SUPPORT_36), "c'est le F1 micro, qui vaut l'accuracy": np.trace(SPECIES_36) / SPECIES_36.sum(), "c'est le F1 de la precision macro et du recall macro : fais plutôt la moyenne des trois F1": 2 * PRECISION_36.mean() * RECALL_36.mean() / (PRECISION_36.mean() + RECALL_36.mean())})
wb.record("3.6g", np.average(F1_36, weights=SUPPORT_36), decimals=3, mistakes={"c'est le F1 macro : la moyenne pondérée multiplie le F1 de chaque espèce par son support": F1_36.mean(), "c'est une moyenne pondérée par les PRÉDICTIONS : le poids d'une espèce est son support (vrais manchots)": np.average(F1_36, weights=SPECIES_36.sum(axis=0)), "tu as fait la moyenne des F1 arrondis de d : repars des fractions exactes des trois F1 et n'arrondis qu'à la fin": 0.869, "c'est le F1 micro, qui vaut l'accuracy : le F1 pondéré fait la moyenne des F1 des espèces, pondérée par leur support": np.trace(SPECIES_36) / SPECIES_36.sum()})
wb.record("3.6h", np.trace(SPECIES_36) / SPECIES_36.sum(), decimals=3, mistakes={"c'est le F1 macro : le F1 micro additionne d'abord les TP, FP et FN des trois espèces": F1_36.mean(), "c'est le F1 pondéré : le F1 micro additionne d'abord les TP, FP et FN des trois espèces": np.average(F1_36, weights=SUPPORT_36)})

WB_ANSWER {"decimals": 0, "element_hashes": ["d19327cae38b42b375ba13cf5ab0f2cc70a2884da4049123c09bc951ddbc1dce", "f7aed9c94bc8303520bf02d2d5b98e44d8748a1952e4b703ee1ab879a5f1f075", "8704c4db430097d1a2c9f8dbc29f31dcdf1563e4f9d0c0a1ddd9865820fef4ef"], "hash": "188b1e2b56178967fcf335600ec514cb4337ba137ade1727571c9905aa9282be", "id": "3.6a", "integer": true, "kind": "array", "mistakes": {"a888f759c7faa7129457f4dffa53a047ff83f5d6b82aa35face7494849e25423": "ce sont les nombres de PRÉDICTIONS de chaque espèce (totaux des colonnes) : le support compte les vrais manchots de chaque espèce (totaux des lignes)"}, "shape": [3]}
📝 Ex 3.6a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 3, "element_hashes": ["8d158ad257b1b7f16f393583759bbffa83022a2b7efd7e38d8f422b0bb643613", "f3a1ea26524eb5509094192f4f7d2eb3a30c02782e05eee762dcded029d3c773", "fb981da27dfc503d85a6adeb672dac4678178efe973f62f0d85dfb21a66b8dad"], "hash": "401b6a143910d1031f38fa33c8313b131c6302d8ae4101bfab94ee6323d638

**Ex 3.7 — Le test « fiable à 99 % » dans une ville à 1 % de malades**

In [8]:
wb.record("3.7a", round(50_000 * 0.01))
wb.record("3.7b", round(TP_37), mistakes={"c'est le nombre de malades : le test n'en détecte que 99 %": round(50_000 * 0.01), "c'est 99 % de toute la ville (et aussi le nombre de personnes saines) : les vrais positifs sont les MALADES que le test détecte, 99 % des seuls malades": round(0.99 * 50_000)})
wb.record("3.7c", round(FN_37), mistakes={"c'est le nombre de faux positifs : un faux négatif est un MALADE déclaré négatif": round(FP_37), "c'est 1 % de toute la ville (ou le nombre de malades) : les faux négatifs sont 1 % des seuls MALADES": round(0.01 * 50_000)})
wb.record("3.7d", round(FP_37), mistakes={"le taux de 1 % de faux positifs s'applique aux 49 500 personnes SAINES, pas à toute la ville": round(0.01 * 50_000), "c'est le nombre de faux négatifs : un faux positif est une personne SAINE déclarée positive": round(FN_37)})
wb.record("3.7e", round(TN_37), mistakes={"les vrais négatifs sont 99 % des personnes SAINES (49 500), pas de toute la ville": round(0.99 * 50_000)})
wb.record("3.7f", TP_37 / (TP_37 + FP_37), decimals=2, mistakes={"c'est la sensibilité P(positif | malade) : la precision est P(malade | positif), divise TP par le nombre de positifs": 0.99})
wb.record("3.7g", TN_37 / (TN_37 + FN_37), decimals=4, mistakes={"c'est la spécificité P(négatif | sain) : la NPV divise TN par le nombre de résultats négatifs": 0.99})
wb.record("3.7h", (TP_37 + TN_37) / 50_000, decimals=2, mistakes={"c'est la precision : l'accuracy compte les bonnes réponses parmi TOUTES les personnes": TP_37 / (TP_37 + FP_37), "c'est la NPV : l'accuracy compte les bonnes réponses, TP + TN, parmi les 50 000 habitants": TN_37 / (TN_37 + FN_37)})
wb.record("3.7i", screening(50_000, 0.002, 0.99, 0.99)[0] / (screening(50_000, 0.002, 0.99, 0.99)[0] + screening(50_000, 0.002, 0.99, 0.99)[2]), decimals=3, mistakes={"c'est la sensibilité : la precision dépend aussi de la prévalence ; refais l'arbre avec 100 malades": 0.99, "deux causes possibles : les faux positifs sont 1 % des personnes SAINES de cette ville, pas de tous ses habitants ; ou bien tu as tronqué au lieu d'arrondir au plus proche": 99 / (99 + 500)})
wb.record("3.7j", screening(50_000, 0.10, 0.99, 0.99)[0] / (screening(50_000, 0.10, 0.99, 0.99)[0] + screening(50_000, 0.10, 0.99, 0.99)[2]), decimals=3, mistakes={"c'est la sensibilité : la precision dépend aussi de la prévalence ; refais l'arbre avec 5 000 malades": 0.99, "les faux positifs sont 1 % des personnes SAINES de cette ville, pas de tous ses habitants": 4950 / (4950 + 500)})
wb.record("3.7k", BOOK[3] / (BOOK[3] + BOOK[2]), decimals=2, mistakes={"c'est la NPV : la spécificité divise TN par le nombre de personnes SAINES, TN + FP": BOOK[3] / (BOOK[3] + BOOK[1]), "ce test n'est pas celui de la partie 1 : calcule TN / (TN + FP) avec les comptages du livre": 0.99, "tu as divisé TN par toute la ville : la spécificité divise par les personnes SAINES, TN + FP": BOOK[3] / 10_000})
wb.record("3.7l", BOOK[3] / (BOOK[3] + BOOK[1]), decimals=4, mistakes={"c'est la spécificité : la NPV divise TN par le nombre de résultats NÉGATIFS, TN + FN": BOOK[3] / (BOOK[3] + BOOK[2])})

WB_ANSWER {"hash": "924dcb0782396abf18adb2ac5b6714d49c364aeb63914c93ecf2dfcc6f414157", "id": "3.7a", "kind": "int", "magnitude": 2, "sign": 1}
📝 Ex 3.7a : réponse enregistrée (int).
WB_ANSWER {"hash": "4f0eba28dea861bd373624b27c8346f98817c333b1efa94328c63757712a6078", "id": "3.7b", "kind": "int", "magnitude": 2, "mistakes": {"b426298b3ef8ed44bc79bc1c0bee63a690d9e45a9b00000ca78d9ede6471a8c2": "c'est 99 % de toute la ville (et aussi le nombre de personnes saines) : les vrais positifs sont les MALADES que le test détecte, 99 % des seuls malades", "b9e1d701d4d5e8377222208591fd69b1364a346dd5a4d6885350360e9007631e": "c'est le nombre de malades : le test n'en détecte que 99 %"}, "sign": 1}
📝 Ex 3.7b : réponse enregistrée (int).
WB_ANSWER {"hash": "59fa531013720995b524f297eb6d723e66ef48a77a5b8508a38453d21a1d214b", "id": "3.7c", "kind": "int", "magnitude": 0, "mistakes": {"d9b09c50020235c2f3781ca5ae5ff2ba4c798e26755c66012ff9a62edcb87c5e": "c'est le nombre de faux positifs : un faux négatif est 

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une probabilité conditionnelle dans une table, et dire laquelle de $P(A \mid B)$ ou $P(B \mid A)$ répond à une question ?
2. Sais-tu construire une matrice de confusion et choisir entre precision et recall selon le coût des erreurs ?
3. Sais-tu expliquer pourquoi un bon test donne surtout des faux positifs quand la maladie est rare ?

**Pour aller plus loin** : les pages du *Machine Learning Crash Course* de Google et le guide « Metrics and scoring » de scikit-learn, cités dans la fiche. La suite : le ch. 4 (règle de Bayes), qui passe de $P(\text{positif} \mid \text{malade})$ à $P(\text{malade} \mid \text{positif})$ ; `mylearn.metrics` servira dans tous les chapitres suivants.

*Les exercices de code 3.12 à 3.29 (parties A à D, ta librairie `mylearn.metrics`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 3` pour obtenir le notebook complet.*